### <u> Generate a local "_Stars Appearing_" sequence: sonification + animation </u>

This builds the "_Stars Appearing_" piece from the "_Audible Universe_" planetarium
show for **any site and any night**, and renders a matching animation to go with it.

The sky is computed with `skyfield` from the _Hipparcos_ catalogue, sonified with
`strauss`, and animated as an **equirectangular** (360&deg; &times; 180&deg;)
panorama. A planetarium dome master comes from letting `ffmpeg`'s `v360` filter
reproject that panorama to fisheye &mdash; we never render fisheye ourselves.

The important idea: the animation does **not** work out for itself when each star
appears. It reads the timings straight out of the rendered sonification, so sound
and picture cannot drift apart.

The `.py` beside this notebook holds the things that are not `strauss` &mdash; the
sky, from `skyfield`, and the animation and its backdrop, from `numpy` and
`ffmpeg`. The sonification itself is not hidden away in there: it happens here, in
the open, with a `strauss` **style** carrying the musical recipe.

> Want the movie rather than the tour? `StarsAppearingColab.ipynb` beside this one
> is the same run behind a settings form: fill it in, hit **Run all**.

#### Requirements

Beyond `strauss` itself you need `skyfield`, and a working `ffmpeg` on your `PATH`.
The first run downloads the _Hipparcos_ catalogue (~50 MB), the DE421 ephemeris
(~17 MB) and NASA's star map (~36 MB) into a cache directory; later runs reuse them.

The next cell does nothing at all unless it finds itself on _Google_ `Colab`, where
it installs `strauss` and fetches the `StarsAppearingLocal.py` module beside this
notebook. It installs `strauss` from the repository rather than from `PyPI`, for
in-development features. Both steps are skipped if they have already run, so
re-running after changing a setting is quick.

In [ ]:
import sys

# Where the pieces come from. These two are the only thing to change if this
# notebook moves to a different repository or branch.
STRAUSS_REF = ("git+https://github.com/james-trayford/strauss.git"
               "@local_stars_appearing_animation")
HELPER_URL = ("https://raw.githubusercontent.com/Audio-Universe/"
              "sonified-night-sky/main/StarsAppearingLocal.py")

if "google.colab" in sys.modules:
    from pathlib import Path

    # both steps are skipped once they are done, so a second `Run all` after
    # changing a setting does not pay for the install again
    try:
        import skyfield, strauss
    except ImportError:
        !pip install --quiet "strauss @ {STRAUSS_REF}" skyfield==1.53

    if not Path("StarsAppearingLocal.py").exists():
        !wget --quiet -O StarsAppearingLocal.py "{HELPER_URL}"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import strauss

from StarsAppearingLocal import (Config, observed_sky, sky_panorama, star_frame,
                                 chosen_style, sonified_events, facing_degrees,
                                 write_videos, show_videos, suite_samples,
                                 restyle, CARDINALS, NIGHT_HARP_NOTES, SOUNDS)

### <u> Chosen properties </u>

Change these and re-run. The defaults describe the **Sherwood Observatory** site
looking south, at a small size so the whole notebook runs in a reasonable time.

In [ ]:
cfg = Config(
    # -- where and when --
    latitude=53.1143737,          # +ve north
    longitude=-1.2219389,         # +ve *east*, so 1.22 degrees west is -1.22
    date_time="2026-09-19 19:00:00",
    time_zone="Europe/London",
    facing="S",                   # centre of the panorama and of the stereo image
    mag_limit=4.,                 # higher includes more, dimmer stars

    # -- the sound --
    duration=45.0,
    system="stereo",              # 'mono', 'stereo', '5.1', '7.1', ...

    # -- the picture --
    # 'fast_preview' is 512x256 and 'preview' 1024x512; 'high' is 4096x2048
    # and 'full' 8192x4096, the native sizes of the two star maps, so at those
    # two a map's pixels are used as they are rather than resampled. 'full'
    # is the one that domes out at 4096 a side. All four are 2:1, the shape of
    # a 360 x 180 degree panorama; give a (width, height) pair of your own and
    # the sky is stretched to fill it.
    size="preview",
    fps=15,
    sky_exposure=0.75,            # raise to bring the Milky Way up
    horizon=False,                # black out everything below the horizon

    # -- outputs --
    output="both",                # 'panorama', 'dome', or 'both'
    outdir="stars_appearing_preview",
)

# The instrument and chord. 'Night Harp' takes both from the Sonification
# Suite's style of that name; 'Glockenspiel' is the sound of the original
# planetarium piece. See "The sound design", below.
sound = "Night Harp"

# For the real thing, closer to what a planetarium would use:
#
#     cfg = Config(mag_limit=5, duration=60, system="5.1", size="full", fps=30,
#                  horizon=True, outdir="stars_appearing_full")
#
# `background` is not set above: it defaults to "auto", which renders a panorama
# to match every setting here (see "The sky to draw it on"). Pass a path to
# `Config` to use an image of your own instead, or `None` for a black sky.
print(f"{cfg.width} x {cfg.height}, writing {cfg.output}")

### <u> The sky </u>

`skyfield` gives the altitude and azimuth of every catalogue star as seen from the
chosen site at the chosen instant. The catalogue is cut down by magnitude *before*
positions are computed, which is the difference between transforming ~118,000 stars
and ~1,500.

In [ ]:
sky = observed_sky(cfg)
print(f"{len(sky)} stars brighter than magnitude {cfg.mag_limit} above the horizon")
sky.head()

A quick look at what we are about to hear, as it would appear on the
panorama &mdash; the facing direction in the middle, the horizon along the bottom.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_facecolor("#0b0c15")

x = (sky["az"] - facing_degrees(cfg.facing) - 180) % 360

ax.scatter(x, sky["alt"], s=40 * 10 ** (-0.2 * sky["magnitude"]),
           c=sky["bv"], cmap="RdYlBu_r", vmin=-1.5, vmax=2.5, lw=0)
ax.set_xticks([(360 * i / 16 - facing_degrees(cfg.facing) - 180) % 360
               for i in range(16)])
ax.set_xticklabels(CARDINALS, fontsize=8)
ax.set_xlim(0, 360)
ax.set_ylim(0, 90)
ax.set_xlabel("compass direction")
ax.set_ylabel("altitude [deg]")
ax.set_title(f"{len(sky)} stars over {cfg.latitude:.2f}, {cfg.longitude:.2f} "
             f"at {cfg.date_time}")
plt.show()

### <u> The sky to draw it on </u>

NASA's [*Deep Star Maps 2020*](https://svs.gsfc.nasa.gov/4851/) are all-sky
equirectangular images in **celestial** coordinates. `sky_panorama` turns one into
the view from where you are standing, for the same site, instant and facing as
everything above &mdash; so the star pulses land on the stars already drawn in it,
and there is no background image to supply by hand.

Increase `sky_exposure` to brighten the Milky Way, lower it to darken the sky and
let the pulses carry the picture.

In [ ]:
background = sky_panorama(cfg)
print(background)

plt.figure(figsize=(14, 14 * cfg.height / cfg.width))
plt.imshow(plt.imread(background))
plt.axis("off")
plt.title(f"the sky over {cfg.latitude:.2f}, {cfg.longitude:.2f} "
          f"at {cfg.date_time}, facing {cfg.facing}")
plt.show()

### <u> The sonification </u>

The recipe lives in a `strauss` **style**, `stars_appearing`, rather than in this
notebook: which notes are available, that a glockenspiel plays them, and the long
decay that makes the piece shimmer as notes pile into each other. Everything the
style does is declared rather than coded, and you can read it:

In [ ]:
strauss.get_style("stars_appearing", print_style=True);

What the style *cannot* do is the part that depends on where you are standing, so
that stays here, in the open. `star_frame` prepares six columns, and the names must
match the `input:` names in the style &mdash; `sonify` matches a `DataFrame`'s
columns to the style by name:

- **`magnitude`** decides *when* each star sounds: brightest first, as the sky
  darkens and your eyes adjust.
- **`colour`** picks the note from the chord, inverted by the style so that blue
  stars take the high notes.
- **`azimuth`** is the one that is easy to get backwards. `strauss` measures azimuth
  **anticlockwise from straight ahead**, while astronomical azimuth runs **clockwise
  from north**, so facing direction *minus* star azimuth is the right way round.
- **`polar`** is measured from the zenith down, not from the horizon up.
- **`volume`** quietens the dimmer stars. They are far more numerous, so without
  this the piece grows steadily louder as it goes.
- **`pitch_shift`** detunes each note very slightly, so that the many stars sharing
  a note do not phase against one another.

In [ ]:
frame = star_frame(sky, cfg)
frame.head()

### <u> The sound design </u>

A different *sound* is a change of style rather than a change of code. The
*Sonification Suite* keeps a design of its own for this piece, "Night Harp", and
its two audible parts carry across as they are: the Suite stores each instrument
as a directory of `.wav` files named by the note each one sounds, which is exactly
what `strauss`'s `Sampler` expects, and its chord is just a list of notes.

`chosen_style` puts the two together. `restyle` copies `stars_appearing` and
replaces only `generator.sample` and `notes`, leaving every mapping above
untouched &mdash; so the piece still sounds one note per star, and still carries
the same data. Longhand, for `sound = "Night Harp"`, it is:

```python
style = restyle("stars_appearing",
                sample=suite_samples("Harp", cfg.cache),
                notes=NIGHT_HARP_NOTES,
                out_path=cfg.outdir / "stars_appearing_harp.yml")
```

In [ ]:
style = chosen_style(sound, cfg)
print(f"sonifying with: {style}\n")

# a restyled style is written out as a file, and is worth a look - the diff
# against the printout above is the whole of the sound design
if Path(str(style)).suffix == ".yml":
    print(Path(style).read_text())

Now sonify it. `source_names` labels each event with its star, so the table below
can be joined back to the catalogue, and `angle_unit="degrees"` is what makes that
table report real degrees rather than raw 0-1 fractions &mdash; without it the
animation would be driven by nonsense.

In [ ]:
strauss.sonify(frame, style=style, channels=cfg.system,
               duration=cfg.duration, angle_unit="degrees",
               source_names=list(frame.index))

strauss.display()

### <u> What sounded, and when </u>

This is the join between sound and picture. `get_table()` reports what is actually
heard &mdash; the time of each note in seconds, the note itself, and the star's
angles in degrees &mdash; and the animation reads its timings from here rather than
working them out again.

The table carries its units in a second column level, which is for reading rather
than for arithmetic. `sonified_events` drops to the plain names, and picks up the
magnitude and colour each pulse is drawn with from the frame we sonified.

In [ ]:
display(strauss.get_table().head(10))

events = sonified_events(frame)
events.head()

### <u> The animation </u>

Each star is a pulse that swells and fades as its note sounds. Frames are generated
as raw `RGBA` and piped straight into `ffmpeg`, which lays them over the background,
muxes the audio, and writes the finished video &mdash; nothing touches the disk in
between.

A star is only drawn while it is bigger than half a pixel, which for these
envelopes is around a second, so only the handful of stars actually alive in each
frame get any work done on them.

`output` above chooses the format. With `both`, two videos come out of a single
pass of the frame generator:

- the **panorama**, equirectangular, 360&deg; across by 180&deg; high
- the **dome master**, the same pixels reprojected to a 180&deg; fisheye by
  `ffmpeg`'s `v360` filter and tilted so the zenith lands in the centre of the
  dome. The facing direction ends up at the bottom of the image, which is the
  usual dome-master orientation.

In [ ]:
audio = cfg.outdir / "stars_appearing.wav"
cfg.outdir.mkdir(parents=True, exist_ok=True)
strauss.save(str(audio))

# with no targets given, `cfg.output` decides which videos get written
videos = write_videos(cfg, events, audio, sky=background)

In [ ]:
show_videos(videos)

### <u> Tidying up, and going bigger </u>

`strauss.close()` finishes with this figure, so that a re-run starts clean rather
than adding a second sonification alongside the first.

Rendering time is dominated by frame size. The preview settings here take a few
seconds. A `high` 4096 &times; 2048, 60-second sequence at 30 fps in `5.1`, giving
both the panorama and the dome master, can take a while to run (hardware and
environment dependent), and `full` is four times the pixels again.

Render at preview size while you are still choosing a site and a night, then raise
`size` and `fps` for the final pass - `full` only where the dome really is 4096
a side.

In [ ]:
strauss.close()